In [1]:
import torch
import collections
from typing import Any, Tuple



In [2]:
class UnigramSampler:
    """单个单词的采样
    """

    def __init__(self, corpus:list, power=0.75, sample_size:int=2):
        r"""对 corpus 进行概率分布的统计，
            并且对每个概率进行 power 次方的处理，
            以增加低频条目被抽中的概率
            公式为

            $$P'(w_i) = \frac{P(w_i)^{0.75}}{\sum_{j}^{n} P(w_j)^{0.75}}$$

        Args:
            corpus (list): 数据列表。
            power (float, optional): 每个概率进行次方 Defaults to 0.75.
            sample_size (int, optional): _description_. Defaults to 2.
        """

        self.sample_size = sample_size
        self.vocab_size = None
        self.word_p = None

        # 统计每个条目的数量。
        counts = collections.Counter()
        for word_id in corpus:
            counts[word_id] += 1

        vocab_size = len(counts)
        self.vocab_size = vocab_size

        # 得到每个条目的原始概率
        self.word_p = torch.zeros(vocab_size)
        for i in range(vocab_size):
            self.word_p[i] = counts[i]

        # 计算每个条目的修正概率
        self.word_p = self.word_p ** power
        self.word_p /= torch.sum(self.word_p)

    def get_negative_sample(self, target:torch.Tensor) -> torch.Tensor:
        """根据目标列表对特征进行负采样

        Args:
            target (torch.Tensor): 目标列表

        Returns:
            torch.Tensor: 根据目标列表返回的负采样矩阵，形状为(batch_size, sample_size)
        """
        batch_size = target.shape[0]
        negative_sample = torch.multinomial(self.word_p.expand(batch_size, -1), self.sample_size, replacement=True)

        return negative_sample

In [3]:
class EmbeddingFunction(torch.autograd.Function):
    """自定义的 Embedding 算子，包含前向和反向传播的底层数学逻辑"""

    @staticmethod
    def forward(ctx: Any, W: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """实现 Embedding 层的正向传播，从权重矩阵中提取特定单词对应的行向量

        Args:
            ctx (Any): 上下文对象，用于存储反向传播需要的信息
            W (torch.Tensor): 权重矩阵，形状为 (V, H)，V为词汇量，H为隐藏层大小
            idx (torch.Tensor): 单词的索引张量

        Returns:
            torch.Tensor: 提取出的对应的分布式表示(词向量)
        """
        # 保存反向传播需要的信息
        ctx.save_for_backward(idx)
        ctx.W_shape = W.shape

        # 实现提取逻辑，并返回结果
        return W[idx]

    @staticmethod
    def backward(ctx: Any, grad_output: torch.Tensor) -> Tuple[torch.Tensor, None]:
        """实现 Embedding 层的反向传播

        Args:
            ctx (Any): 上下文对象，包含前向传播时保存的 idx 和 W_shape
            grad_output (torch.Tensor): 后面层传回来的梯度，形状与 forward 的输出一致

        Returns:
            Tuple[torch.Tensor, None]:
            - grad_W: 对权重矩阵 W 的梯度
            - None: idx 是离散的索引，不需要梯度，返回 None
        """
        # 反向传播，重复的idx值需要相加
        idx = ctx.saved_tensors[0]

        dW = torch.zeros(ctx.W_shape)
        dW.index_add_(dim=0, index=idx, source=grad_output)

        return dW, None

class Embedding(torch.nn.Module):
    """Embedding 网络层，用于将单词 ID 批量转换为对应的分布式表示"""

    def __init__(self, W: torch.Tensor):
        """初始化 Embedding 层

        Args:
            W (torch.Tensor): 初始化的权重矩阵，形状为 (V, H)
        """
        super().__init__()
        # 将传入的张量注册为模型的参数，使其可以被优化器更新
        self.W = torch.nn.Parameter(W)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """Embedding 层的前向传播调用

        Args:
            idx (torch.Tensor): 输入的单词 ID 张量

        Returns:
            torch.Tensor: 对应的词向量张量
        """
        return EmbeddingFunction.apply(self.W, idx)

In [4]:
class BatchDotFunction(torch.autograd.Function):
    """处理 h 和 target_W 批量内积的自定义算子"""

    @staticmethod
    def forward(ctx: Any, h: torch.Tensor, target_W: torch.Tensor) -> torch.Tensor:
        """
        Args:
            h (torch.Tensor): 中间层神经元张量，形状为 (batch_size, hidden_size)
            target_W (torch.Tensor): 提取出的词向量张量，形状为 (batch_size, hidden_size)

        Returns:
            torch.Tensor: 内积得分，形状为 (batch_size,)
        """
        # 1. 保存 h 和 target_W 供反向传播使用
        ctx.save_for_backward(h, target_W)

        # 2. 正向传播：计算对应元素乘积，并按行求和 (对应书中 np.sum(target_W * h, axis=1))
        return torch.sum(target_W * h, dim=1)


    @staticmethod
    def backward(ctx: Any, grad_output: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            grad_output (torch.Tensor): 从 SigmoidWithLoss 传回的梯度，形状 (batch_size,)

        Returns:
            Tuple[torch.Tensor, torch.Tensor]:
            分别对应 h 和 target_W 的梯度(返回值的形状均为(batch_size, hidden_size))
        """
        # 1. 取出保存的 h 和 target_W
        h, target_W = ctx.saved_tensors

        # 2. 请在此处实现反向传播：重塑 grad_output 并分别计算 grad_h 和 grad_target_W
        grad_h = grad_target_W = None
        grad_output = grad_output.view(-1, 1)

        if ctx.needs_input_grad[0]:
            grad_h = grad_output * target_W

        if ctx.needs_input_grad[1]:
            grad_target_W = grad_output * h

        return grad_h, grad_target_W

class EmbeddingDot(torch.nn.Module):
    """将 Embedding 层和内积运算合并的层"""

    def __init__(self, W: torch.Tensor):
        """初始化 EmbeddingDot 层

        Args:
            W (torch.Tensor): 初始化的输出侧权重矩阵，形状为 (V, H)
        """
        super().__init__()
        # 复用刚才的 Embedding 层！
        self.embed = Embedding(W)

    def forward(self, h: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """EmbeddingDot 层的前向传播调用

        Args:
            h (torch.Tensor): 中间层神经元张量，形状为 (batch_size, H)
            idx (torch.Tensor): 目标词的索引张量，形状为 (batch_size,)

        Returns:
            torch.Tensor: 对应目标词的内积得分
        """

        target_W = self.embed(idx)
        return BatchDotFunction.apply(h, target_W)

In [5]:
class SigmoidWithLossFunction(torch.autograd.Function):
    """包含 Sigmoid 激活与二分类交叉熵损失反向推导逻辑的自定义算子"""

    @staticmethod
    def forward(ctx: Any, x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        r"""实现 Sigmoid 激活和交叉熵损失的计算。

        即两个公式的实现：
        $$
        y=\frac{1}{1+e^{-x}}
        $$
        $$L = - \left[ t \cdot \log(y) + (1-t) \cdot \log(1 - y) \right]$$

        其中 t 为真实结果，y 为预测结果。

        Args:
            ctx (Any): 上下文对象，用于存储反向传播所需数据。
            x (torch.Tensor): 预测得分，形状为 (batch_size,)。
            target (torch.Tensor): 正确解标签 (0 或 1)，形状为 (batch_size,)。

        Returns:
            torch.Tensor: 计算得到的标量总损失。
        """

        # 为了防止 log(0) 导致数值不稳定，请在 y 内部加上一个极小数（如 1e-7）
        y = 1 / (1 + torch.exp(-x))
        loss = torch.sum(-(target * torch.log(y + 1e-7) + (1 - target) * torch.log(1 - y + 1e-7)))

        # 保存 y 和 target 供反向传播使用
        ctx.save_for_backward(y, target)

        return loss

    @staticmethod
    def backward(ctx: Any, grad_output: torch.Tensor) -> Tuple[torch.Tensor, None]:
        """实现交叉熵与 Sigmoid 组合后的极简反向传播。

        Args:
            ctx (Any): 上下文对象，包含前向传播时保存的 y 和 target。
            grad_output (torch.Tensor): 标量损失传回的梯度，通常为 1.0 (张量)。

        Returns:
            Tuple[torch.Tensor, None]:
            - grad_x: 对输入得分 x 的梯度。
            - None: target 是真实的离散标签，不需要求导，返回 None。
        """
        # 1. 取出前向传播保存的张量
        y, target = ctx.saved_tensors

        # 2. 请在此处根据书上推导的公式 (y - t) 计算梯度
        # 别忘了乘上链式法则传进来的 grad_output

        return grad_output * (y - target), None

class SigmoidWithLoss(torch.nn.Module):
    """Sigmoid 与二分类交叉熵损失函数的组合网络层"""

    def __init__(self):
        """初始化 SigmoidWithLoss 层"""
        super().__init__()

    def forward(self, x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """SigmoidWithLoss 层的前向传播调用

        Args:
            x (torch.Tensor): 预测得分，形状为 (batch_size,)。
            target (torch.Tensor): 正确解标签 (0 或 1)，形状为 (batch_size,)。

        Returns:
            torch.Tensor: 计算出的标量总损失。
        """
        return SigmoidWithLossFunction.apply(x, target)

In [6]:
class NegativeSamplingLoss(torch.nn.Module):
    def __init__(self, W: torch.Tensor, corpus: list, power: float = 0.75, sample_size: int = 5):
        super().__init__()
        self.sample_size = sample_size
        # 调用 get_negative_sample(target) 会返回形状为 (batch_size, sample_size) 的负例张量
        self.sampler = UnigramSampler(corpus, power, sample_size)

        # 只需实例化一个 EmbeddingDot，它会自动复用底层的 W
        self.embed_dot = EmbeddingDot(W)

        # PyTorch 内置的 Sigmoid + 交叉熵，reduction='sum' 表示求和
        self.loss_fn = SigmoidWithLoss()

    def forward(self, h: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Args:
            h (torch.Tensor): 中间层神经元，形状 (batch_size, hidden_size)
            target (torch.Tensor): 正例(正确的单词ID)，形状 (batch_size,)

        Returns:
            torch.Tensor: 正例和所有负例的总损失
        """
        batch_size = target.shape[0]

        # 1. 采样负例，形状为 (batch_size, sample_size)
        negative_sample = self.sampler.get_negative_sample(target)

        # --- 正例的损失计算 ---
        # 提示：
        # 1. 用 self.embed_dot 计算正例得分
        positive_score = self.embed_dot(h, target)
        # 2. 创建一个全为 1 的标签张量
        positive_target = torch.ones_like(positive_score)
        # 3. 用 self.loss_fn 计算正例损失
        total_loss = self.loss_fn.forward(positive_score, positive_target)

        # --- 负例的损失计算 ---
        # 提示：
        # 1. 创建一个全为 0 的标签张量
        negative_target = torch.zeros_like(positive_score)
        # 2. 遍历 self.sample_size 次，每次取出 negative_sample 的一列
        for i in range(self.sample_size):
             # 提取第 i 列，形状变为 (batch_size,)
            negative_sample_idx = negative_sample[:, i]
            negative_score = self.embed_dot(h, negative_sample_idx)
            # 3. 计算得分并求损失，累加到总损失中
            total_loss += self.loss_fn.forward(negative_score, negative_target)
        # 返回总损失
        return total_loss

In [7]:
# ==========================================
# 1. 准备玩具数据
# ==========================================
vocab_size = 7
hidden_size = 3
batch_size = 3
sample_size = 2

# 模拟一个极小的语料库
corpus = [0, 1, 2, 3, 4, 1, 2, 3]

# 模拟输出侧权重矩阵 W_out，形状 (7, 3)
W_out = torch.arange(21, dtype=torch.float32).reshape(vocab_size, hidden_size)

# 模拟中间层传过来的神经元特征 h，形状 (3, 3)
# 注意：我们设置 requires_grad=True，以便一会测试反向传播！
h = torch.tensor([
    [ 0.5,  0.1, -0.2],
    [-0.3,  0.8,  0.0],
    [ 0.0, -0.1,  0.6]
], requires_grad=True)

# 模拟这一批次的正确目标词 (正例)
target = torch.tensor([1, 3, 0])

# ==========================================
# 2. 实例化网络层并运行
# ==========================================
print("🚀 开始测试 NegativeSamplingLoss...")

# 实例化负采样损失层 (假设采样 2 个负例)
loss_layer = NegativeSamplingLoss(W_out, corpus, power=0.75, sample_size=sample_size)

# 【测试正向传播】
loss = loss_layer(h, target)
print(f"\n✅ 前向传播成功！算出的总损失 Loss: {loss.item():.4f}")

# 【测试反向传播】
loss.backward()
print("\n✅ 反向传播成功！")
print(f"👉 中间层 h 收到的梯度 (形状应为 {h.shape}):\n{h.grad}")
print(f"👉 权重 W_out 收到的梯度 (形状应为 {W_out.shape}):\n{loss_layer.embed_dot.embed.W.grad}")

🚀 开始测试 NegativeSamplingLoss...

✅ 前向传播成功！算出的总损失 Loss: 16.7424

✅ 反向传播成功！
👉 中间层 h 收到的梯度 (形状应为 torch.Size([3, 3])):
tensor([[ 4.4783,  5.5057,  6.5331],
        [ 2.6819,  4.2758,  5.8697],
        [14.7827, 16.4630, 18.1433]])
👉 权重 W_out 收到的梯度 (形状应为 torch.Size([7, 3])):
tensor([[ 0.0058,  0.6195, -0.2350],
        [-0.4172,  0.6051,  0.6163],
        [ 0.4455,  0.0891, -0.1782],
        [ 0.0015, -0.0040,  0.0000],
        [ 0.0000, -0.0999,  0.5995],
        [ 0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000]])
